In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.gridspec as gridspec
import pandas as pd
import seaborn as sns

import os
from tqdm import tqdm

In [2]:
POSTS_DIR = 'data/processed_data'
PLOTS_DIR = 'plots/'
os.makedirs(PLOTS_DIR, exist_ok=True)

files_name = os.listdir(POSTS_DIR)

post_1_ind = 20
post_2_ind = 12
print(files_name[post_1_ind], files_name[post_2_ind])

post_1_data = pd.read_csv(os.path.join(POSTS_DIR,files_name[post_1_ind]))
post_2_data = pd.read_csv(os.path.join(POSTS_DIR,files_name[post_2_ind]))

AKTAU.csv BAKU.csv


In [3]:
import json
with open('configs/post_dict.json', 'r') as f:
    posts_dir = json.load(f)

In [4]:
post_ind = [posts_dir[item] for item in files_name]

In [5]:
post_1_data['datetime'] = pd.to_datetime(post_1_data['datetime'])
post_1_data['sea_level'] = pd.to_numeric(post_1_data['sea_level'], errors='coerce')

post_2_data['datetime'] = pd.to_datetime(post_2_data['datetime'])
post_2_data['sea_level'] = pd.to_numeric(post_2_data['sea_level'], errors='coerce')

/tmp/ipykernel_21264/897010239.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  post_1_data['datetime'] = pd.to_datetime(post_1_data['datetime'])
/tmp/ipykernel_21264/897010239.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  post_2_data['datetime'] = pd.to_datetime(post_2_data['datetime'])


In [6]:
width = 75
height = 65

plt.rcParams.update({
    'font.family': 'sans-serif',  # 'sans-serif' 'serif'
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'lines.linewidth': 0.8,
    'axes.linewidth': 0.6,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,

    'grid.linewidth': 0.4,
    'grid.alpha': 0.7,   
    'grid.linestyle': '--',
})

border_params_set_1 = {
    'left': 0.14,
    'right': 0.99,
    'bottom': 0.1,
    'top': 0.99
}

border_params_set_2 = {
    'left': 0.18,
    'right': 0.99,
    'bottom': 0.13,
    'top': 0.99
}

In [7]:
def plot_type_1(data_1: pd.DataFrame, data_2: pd.DataFrame, name_1: str, name_2: str, save: bool=False) -> None:
    '''
    Make plot type 1
    :params:
        data_1: pd.DataFrame - data for 1-st post/altimetry 
        data_2: pd.DataFrame - data for 2-nd post/altimetry 
        name_1: str - name for 1-st post/altimerty
        name_2: str - name for 2-nd post/altimerty 
        save: bool (default=False) - flag, if True only save figure without showing, if False only show figure without saving
    '''
    # Prepare figure
    fig = plt.figure(figsize=(width/25.4, height/25.4), dpi=600)
    ax = fig.add_subplot(111)
    fig.subplots_adjust(**border_params_set_1)

    # Start time
    t_0 = max(data_1['datetime'].min(), data_2['datetime'].min())

    # Plot
    ax.plot(data_1[data_1['datetime']>=t_0]['datetime'], data_1[data_1['datetime']>=t_0]['sea_level'], label=name_1)
    ax.plot(data_2[data_2['datetime']>=t_0]['datetime'], data_2[data_2['datetime']>=t_0]['sea_level'], label=name_2)

    # Utils
    ax.legend()
    ax.grid(True)

    if(save):
        os.makedirs(f"{PLOTS_DIR}/{name_1}_{name_2}", exist_ok=True)
        fig.savefig(f'{PLOTS_DIR}/{name_1}_{name_2}/figure_type_1.png', format='png', dpi=600)
    else:
        plt.show()

    plt.close(fig)

In [8]:
plot_type_1(post_1_data, post_2_data, post_ind[post_1_ind], post_ind[post_2_ind], save=True)

In [9]:
from sklearn.linear_model import LinearRegression
from typing import Dict

def plot_type_2(data_1: pd.DataFrame, data_2: pd.DataFrame, name_1: str, name_2: str, save: bool=False) -> None:
    '''
    Make plot type 2
    :params:
        data_1: pd.DataFrame - data for 1-st post/altimetry 
        data_2: pd.DataFrame - data for 2-nd post/altimetry 
        name_1: str - name for 1-st post/altimerty
        name_2: str - name for 2-nd post/altimerty 
        save: bool (default=False) - flag, if True only save figure without showing, if False only show figure without saving
    '''
    # Prepare figure
    fig = plt.figure(figsize=(width/25.4, height/25.4), dpi=600)
    ax = fig.add_subplot(111)
    fig.subplots_adjust(**border_params_set_2)

    # Prepare data
    df_dummy = pd.merge(data_1, data_2, on='datetime', suffixes=('_post1', '_post2')).dropna()

    # Some statistics for visualization
    statistics = stat(df_dummy['sea_level_post2'], df_dummy['sea_level_post1'])
    x_min_ = np.min(df_dummy['sea_level_post2'])
    x_max_ = np.max(df_dummy['sea_level_post2'])
    y_min_ = np.min(df_dummy['sea_level_post1'])
    y_max_ = np.max(df_dummy['sea_level_post1'])

    # Plot based on size
    if((width==75) and (height==65)):
        ax.scatter(df_dummy['sea_level_post2'], df_dummy['sea_level_post1'], s=2.2, color=[0, 191/255, 255/255])
        ax.plot([x_min_, x_max_],[x_min_*statistics['a']+statistics['b'], x_max_*statistics['a']+statistics['b']], linewidth=1.5, label='y=ax+b')
        d_ = y_max_ - y_min_
        delta_ = d_/13
        text_str = f"Entries = {statistics['entries']}\nCORR = {statistics['corr']:.3f}\nRMSE = {statistics['rmse']:.3f}\na = {statistics['a']:.3f}\nb = {statistics['b']:.3f}\nSD = {statistics['sd']:.3f}\nR2 = {statistics['r2']:.3f}\nBias = {statistics['me']:.3f}"
        plt.text(x_min_, y_max_-5.4*delta_, 
                text_str,
                bbox=dict(boxstyle='round,pad=0.1', 
                        facecolor='white', 
                        alpha=0.8, 
                        edgecolor='black',
                        linewidth=0.5), size=7.0)
        plt.legend()
    elif((width==75) and (height==60)):
        ax.scatter(df_dummy['sea_level_post2'], df_dummy['sea_level_post1'], s=2.2)
        ax.plot([x_min_, x_max_],[x_min_*statistics['a']+statistics['b'], x_max_*statistics['a']+statistics['b']], linewidth=1.5, color=[250/255, 0, 0])
    
    plt.xlabel(post_ind[post_2_ind], labelpad=1)
    plt.ylabel(post_ind[post_1_ind], labelpad=1)

    ax.grid(True)

    if save:
        fig.savefig(f'{PLOTS_DIR}/{name_1}_{name_2}/figure_type_2.png', format='png', dpi=600)
    else:
        plt.show()
    plt.close(fig)

def stat(x: np.ndarray, y: np.ndarray) -> Dict:
    '''
    Calculate statistics between two data:
    :params:
        x: np.ndarray - 1-st array
        y: np.ndarray - 2-nd array
    :return:
        statistics: Dict - dictionary with statistics
    '''
    assert len(x)==len(y)
    mu_x = np.mean(x)
    mu_y = np.mean(y)

    m = LinearRegression().fit(x.values.reshape(-1, 1), y.values.reshape(-1, 1))
    a = m.coef_[0,0]
    b = m.intercept_[0]

    me = np.mean(y-x)
    sd = np.sqrt(np.sum((y-x-me)**2)/(len(x)-1))
    rmse = np.sqrt(me**2+sd**2)

    corr = np.mean((x-mu_x)*(y-mu_y))/np.sqrt(np.mean((x-mu_x)**2)*np.mean((y-mu_y)**2))
    r_2 = 1 - np.sum((y - (a*x+b))**2)/np.sum((y - mu_y)**2)

    return {
        'entries': len(x),
        'corr': corr,
        'rmse': rmse,
        'a': a,
        'b': b,
        'sd': sd,
        'r2': r_2,
        'me': me
    }

In [10]:
plot_type_2(post_1_data, post_2_data, post_ind[post_1_ind], post_ind[post_2_ind], save=True)

In [11]:
def plot_type_3(data_1: pd.DataFrame, data_2: pd.DataFrame, name_1: str, name_2: str, save: bool=False) -> None:
    '''
    Make plot type 1
    :params:
        data_1: pd.DataFrame - data for 1-st post/altimetry 
        data_2: pd.DataFrame - data for 2-nd post/altimetry 
        name_1: str - name for 1-st post/altimerty
        name_2: str - name for 2-nd post/altimerty 
        save: bool (default=False) - flag, if True only save figure without showing, if False only show figure without saving
    '''
    # Prepare figure
    fig = plt.figure(figsize=(width/25.4, height/25.4), dpi=600)
    ax = fig.add_subplot(111)
    fig.subplots_adjust(**border_params_set_1)

    # Prepare data
    df_dummy = pd.merge(data_1, data_2, on='datetime', suffixes=('_post1', '_post2')).dropna()

    # Plot
    ax.plot(df_dummy['datetime'], df_dummy['sea_level_post1']-df_dummy['sea_level_post2'])

    # Utils
    ax.grid(True)

    if(save):
        os.makedirs(f"{PLOTS_DIR}/{name_1}_{name_2}", exist_ok=True)
        fig.savefig(f'{PLOTS_DIR}/{name_1}_{name_2}/figure_type_3.png', format='png', dpi=600)
    else:
        plt.show()

    plt.close(fig)

In [12]:
plot_type_3(post_1_data, post_2_data, post_ind[post_1_ind], post_ind[post_2_ind], save=True)